# Transfer Generated Files

Use this notebook after `agent_coding_workflow.ipynb` finishes a sprint run.

It reads `outputs/latest_run.txt`, loads the latest run folder, shows each generated file with its `PASS` or `FAIL` status, and lets you choose which files to copy into the target ASP.NET Core app repo.

Default choices are conservative:

1. `PASS` files default to apply.
2. `FAIL` files default to skip.
3. Files without a saved source are skipped.

The notebook writes `applied_files.json` and `command_logs.jsonl` back into the selected run folder for traceability.


In [1]:
# Step 1: Imports and path setup
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
from datetime import datetime

# The notebook is expected to run from the CPE494-agent-coding-team repo root.
# In VS Code, set the notebook working directory to the repository root if needed.
REPO_ROOT = Path.cwd().resolve()

# If the notebook is opened from notebooks/, move one level up automatically.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.resolve()

TARGET_APP_PATH = (REPO_ROOT.parent / "CPE494-erp-invoice-app-by-ai").resolve()
OUTPUTS_DIR = REPO_ROOT / "outputs"
RUNS_DIR = OUTPUTS_DIR / "runs"
LATEST_RUN_PATH = OUTPUTS_DIR / "latest_run.txt"

print("Agent repo root:", REPO_ROOT)
print("Target app path:", TARGET_APP_PATH)
print("Latest run marker:", LATEST_RUN_PATH)


Agent repo root: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team
Target app path: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-erp-invoice-app-by-ai
Latest run marker: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\latest_run.txt


In [2]:
# Step 2: Utility functions

def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def save_json(path: Path, data: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def get_git_status_short(repo_path: Path) -> str:
    try:
        result = subprocess.run(
            ["git", "status", "--short"],
            cwd=str(repo_path),
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception:
        return "unknown"


def safe_relative_path(file_name: str) -> Path:
    """Return a safe relative path for a generated target file."""
    rel = Path(file_name.replace("\\", "/"))
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"Unsafe file path from task: {file_name}")
    return rel


def read_latest_run_id() -> str:
    if not LATEST_RUN_PATH.exists():
        raise FileNotFoundError(f"Missing latest run marker: {LATEST_RUN_PATH}")
    run_id = LATEST_RUN_PATH.read_text(encoding="utf-8").strip()
    if not run_id:
        raise ValueError(f"Latest run marker is empty: {LATEST_RUN_PATH}")
    return run_id


def load_latest_run_dir() -> Path:
    run_id = read_latest_run_id()
    run_dir = RUNS_DIR / run_id
    if not run_dir.exists():
        raise FileNotFoundError(f"Latest run folder does not exist: {run_dir}")
    return run_dir


In [3]:
# Step 3: Load the latest run
RUN_ID = read_latest_run_id()
RUN_DIR = load_latest_run_dir()
GENERATED_DIR = RUN_DIR / "generated_files"

print("Latest run ID:", RUN_ID)
print("Run folder:", RUN_DIR)
print("Workflow result:", RUN_DIR / "workflow_result.json")


Latest run ID: 2026-05-04_141246_sprint_01
Run folder: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs\2026-05-04_141246_sprint_01
Workflow result: C:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\outputs\runs\2026-05-04_141246_sprint_01\workflow_result.json


In [4]:
# Step 4: Build apply candidates from workflow_result.json

def list_generated_files() -> list[Path]:
    if not GENERATED_DIR.exists():
        return []
    return sorted([p for p in GENERATED_DIR.rglob("*") if p.is_file()])


def load_workflow_results(run_dir: Path) -> list[dict]:
    """Return per-file workflow results if workflow_result.json exists."""
    workflow_result_path = run_dir / "workflow_result.json"
    if not workflow_result_path.exists():
        return []
    data = json.loads(workflow_result_path.read_text(encoding="utf-8"))
    return data.get("results", [])


def latest_attempt_code_path(run_dir: Path, relative_path: Path) -> Path | None:
    """Return the latest saved attempt code for a file, if any."""
    attempt_dir = run_dir / "attempts" / relative_path.as_posix().replace("/", "__")
    attempt_files = sorted(attempt_dir.glob("attempt_*_code.txt"))
    if not attempt_files:
        return None
    return attempt_files[-1]


def apply_feature_group_defaults(candidates: list[dict]) -> list[dict]:
    """Keep Razor page markup and PageModel files together by default."""
    by_path = {candidate["relative_path"].as_posix(): candidate for candidate in candidates}

    for candidate in candidates:
        rel = candidate["relative_path"]
        rel_text = rel.as_posix()
        paired_text = None

        if rel_text.endswith(".cshtml"):
            paired_text = f"{rel_text}.cs"
        elif rel_text.endswith(".cshtml.cs"):
            paired_text = rel_text[:-3]

        if not paired_text or paired_text not in by_path:
            continue

        pair = [candidate, by_path[paired_text]]
        if any(item["status"] != "PASS" for item in pair):
            for item in pair:
                item["default_apply"] = False
                item["default_reason"] = "paired Razor page file did not pass"
                item["feature_group"] = rel_text[:-3] if rel_text.endswith(".cshtml.cs") else rel_text

    return candidates


def build_apply_candidates() -> list[dict]:
    """Build target-file apply candidates with PASS/FAIL and bundle-aware defaults."""
    workflow_results = load_workflow_results(RUN_DIR)
    candidates = []

    if workflow_results:
        for result in workflow_results:
            rel = safe_relative_path(result["file_name"])
            status = result.get("status", "UNKNOWN").upper()
            generated_path = GENERATED_DIR / rel
            if generated_path.exists():
                source = generated_path
                source_kind = "generated_files"
            else:
                source = latest_attempt_code_path(RUN_DIR, rel)
                source_kind = "latest_attempt"

            candidates.append({
                "relative_path": rel,
                "status": status,
                "attempts": result.get("attempts"),
                "default_apply": status == "PASS",
                "default_reason": "status PASS" if status == "PASS" else "status not PASS",
                "source": source,
                "source_kind": source_kind,
            })
        return apply_feature_group_defaults(candidates)

    # Backward-compatible fallback for older runs without workflow_result.json.
    candidates = [
        {
            "relative_path": p.relative_to(GENERATED_DIR),
            "status": "PASS",
            "attempts": None,
            "default_apply": True,
            "default_reason": "generated file without workflow_result.json",
            "source": p,
            "source_kind": "generated_files",
        }
        for p in list_generated_files()
    ]
    return apply_feature_group_defaults(candidates)


candidates = build_apply_candidates()
print(f"Found {len(candidates)} candidate files.")
for candidate in candidates:
    rel = candidate["relative_path"].as_posix()
    source = candidate.get("source")
    source_note = candidate.get("source_kind") if source and source.exists() else "no source"
    default_text = "apply" if candidate["default_apply"] else "skip"
    reason = candidate.get("default_reason", "")
    print(f"- {rel}: {candidate['status']} ({source_note}, default {default_text}; {reason})")


Found 8 candidate files.
- wwwroot/css/zen-green.css: PASS (generated_files, default apply; status PASS)
- wwwroot/css/site.css: PASS (generated_files, default apply; status PASS)
- Pages/Shared/_Layout.cshtml: PASS (generated_files, default apply; status PASS)
- Pages/Shared/_Layout.cshtml.css: PASS (generated_files, default apply; status PASS)
- Pages/Index.cshtml: PASS (generated_files, default apply; status PASS)
- Pages/Index.cshtml.cs: PASS (generated_files, default apply; status PASS)
- Pages/Login.cshtml: FAIL (latest_attempt, default skip; paired Razor page file did not pass)
- Pages/Login.cshtml.cs: PASS (generated_files, default skip; paired Razor page file did not pass)


In [5]:
# Step 5: Apply selected files to the target app repo

def prompt_apply_candidate(candidate: dict) -> bool:
    """Ask whether to apply one file, using Enter to accept the default."""
    rel_text = candidate["relative_path"].as_posix()
    default_text = "y" if candidate["default_apply"] else "n"
    status_text = candidate["status"]
    source = candidate.get("source")
    if source is None or not source.exists():
        print(f"{rel_text} [{status_text}] has no saved source file; skipping.")
        return False

    answer = input(f"Apply {rel_text} [{status_text}]? (y/n, default {default_text}): ").strip().lower()
    if not answer:
        return candidate["default_apply"]
    return answer in {"y", "yes"}


def apply_generated_files_to_target():
    candidates = build_apply_candidates()

    if not candidates:
        print("No generated files or attempts to apply.")
        return []

    print("Review files to apply. Press Enter to accept each default.")
    print("Default is y for PASS files and n for failed files.")

    selected_candidates = []
    skipped_records = []
    for candidate in candidates:
        selected = prompt_apply_candidate(candidate)
        rel_text = candidate["relative_path"].as_posix()
        if selected:
            selected_candidates.append(candidate)
        else:
            skipped_records.append({
                "relative_path": rel_text,
                "status": candidate["status"],
                "default_apply": candidate["default_apply"],
                "default_reason": candidate.get("default_reason"),
                "feature_group": candidate.get("feature_group"),
                "source": str(candidate["source"]) if candidate.get("source") else None,
                "source_kind": candidate.get("source_kind"),
            })
            print(f"Skipped: {rel_text}")

    if not selected_candidates:
        print("No files selected to apply.")
        save_json(RUN_DIR / "applied_files.json", {
            "applied": False,
            "applied_at": now_iso(),
            "run_id": RUN_ID,
            "target_app_path": str(TARGET_APP_PATH),
            "files": [],
            "skipped_files": skipped_records,
        })
        return []

    applied_records = []
    for candidate in selected_candidates:
        src = candidate["source"]
        rel = candidate["relative_path"]
        dest = TARGET_APP_PATH / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest)
        record = {
            "relative_path": rel.as_posix(),
            "status": candidate["status"],
            "source": str(src),
            "source_kind": candidate.get("source_kind"),
            "default_reason": candidate.get("default_reason"),
            "feature_group": candidate.get("feature_group"),
            "destination": str(dest),
            "sha256": file_sha256(dest),
            "applied_at": now_iso(),
        }
        applied_records.append(record)
        print(f"Applied: {rel.as_posix()}")

    save_json(RUN_DIR / "applied_files.json", {
        "applied": True,
        "applied_at": now_iso(),
        "run_id": RUN_ID,
        "target_app_path": str(TARGET_APP_PATH),
        "target_app_repo_status_after_apply": get_git_status_short(TARGET_APP_PATH),
        "files": applied_records,
        "skipped_files": skipped_records,
    })

    print("Done. Review git diff in the target app repo.")
    return applied_records


applied_records = apply_generated_files_to_target()


Review files to apply. Press Enter to accept each default.
Default is y for PASS files and n for failed files.
Skipped: Pages/Login.cshtml
Skipped: Pages/Login.cshtml.cs
Applied: wwwroot/css/zen-green.css
Applied: wwwroot/css/site.css
Applied: Pages/Shared/_Layout.cshtml
Applied: Pages/Shared/_Layout.cshtml.css
Applied: Pages/Index.cshtml
Applied: Pages/Index.cshtml.cs
Done. Review git diff in the target app repo.


In [6]:
# Stop here during normal execution.
raise SystemExit("Normal stop point: can go run in terminal")

SystemExit: Normal stop point: can go run in terminal

c:\Users\CPE KMUTT\OneDrive\Documents\teaching\CPE494\lab-coding-agent\CPE494-agent-coding-team\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Step 6: Build and Git helpers

def run_command(command: list[str], cwd: Path, label: str | None = None) -> subprocess.CompletedProcess:
    started_at = now_iso()
    print("Running:", " ".join(command))
    result = subprocess.run(command, cwd=str(cwd), capture_output=True, text=True, shell=False)

    print("STDOUT:")
    print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    print("Return code:", result.returncode)

    append_jsonl(RUN_DIR / "command_logs.jsonl", {
        "run_id": RUN_ID,
        "label": label or " ".join(command),
        "command": command,
        "cwd": str(cwd),
        "started_at": started_at,
        "finished_at": now_iso(),
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    })

    return result


def dotnet_build():
    return run_command(["dotnet", "build"], TARGET_APP_PATH, label="dotnet_build")


def git_diff():
    return run_command(["git", "diff", "--stat"], TARGET_APP_PATH, label="git_diff_stat")


def git_status():
    return run_command(["git", "status", "--short"], TARGET_APP_PATH, label="git_status_short")


In [ ]:
# Step 7: Build the target ASP.NET Core application.
dotnet_build()


In [ ]:
# Step 8: Inspect Git status and diff summary in the target app repo.
git_status()
git_diff()
